In [1]:
import modeling_pretrain as modeling
import torch
from timm.models import create_model
import torch.nn as nn
from functools import partial

args = {
    # "model_name": 'pretrain_videomae_base_dim512_no_depth_patch16_160',
    "encoder_depth": 16,
    "decoder_depth": 4,
    "epochs": 100,
    "lr": 3e-4,
    "num_workers": 20,
    "attn_type": "local_global",
    "part_win_size": (2, 5, 10),
    "lg_region_size": (2, 5, 10),
    "use_frame_diff_as_target": True,
    "mask_ratio": 0.9,
    "mask_type": "part_window",
    "input_size": 160,
    "batch_size": 32,
    "num_frames": 16,
    "sampling_rate": 4,
    "opt": "adamw",
    "opt_betas": (0.9, 0.95),
    "warmup_epochs": 5,
    "save_ckpt_freq": 10,
    "epochs": 100,
    "log_dir": "./output",
    "output_dir": "./output",
    "lr": 3e-4,
    "drop_path_rate": 0,
    "drop_block_rate": None,
    "lg_first_attn_type": "self",
    "lg_third_attn_type": "cross",
    "lg_attn_param_sharing_first_third": False,
    "lg_attn_param_sharing_all": False,
    "lg_no_second": False,
    "lg_no_third": False,
    "num_workers": 20,
    "attn_type": "local_global",
    "part_win_size": (2, 5, 10),
    "lg_region_size": (2, 5, 10),
    "use_frame_diff_as_target": True,
    "img_size": 160,
    "patch_size": 16,
    "encoder_embed_dim": 512,
    "encoder_num_heads":8,
    "encoder_num_classes":0,
    "decoder_num_classes": 1536,
    "decoder_embed_dim": 384,
    "decoder_num_heads": 6,
    "mlp_ratio": 4,
    "qkv_bias": True,
    "norm_layer": partial(nn.LayerNorm, eps=1e-6),
}
pth_path = "saved/model/pretraining/voxceleb2/videomae_pretrain_base_dim512_local_global_attn_depth16_region_size2510_patch16_160_frame_16x4_tube_mask_ratio_0.9_e100_with_diff_target_server170/checkpoint-49.pth"
state_dict = torch.load(pth_path, map_location='cpu')
model = modeling.PretrainVisionTransformer(**args)
model.load_state_dict(state_dict['model'], strict=True)

/equilibrium/fvilli/miniconda3/envs/pytorch1.7/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


==> Note: Use 'local_global' for compute reduction (lg_region_size=(2, 5, 10),lg_first_attn_type=self, lg_third_attn_type=cross,lg_attn_param_sharing_first_third=False,lg_attn_param_sharing_all=False,lg_no_second=False, lg_no_third=False)
==> Number of local regions: 8 (size=[4, 2, 1])


<All keys matched successfully>

In [2]:
rnd_image = torch.randn(2, 3, 16, 160, 160).to(torch.uint8) # B, C, T, H, W
# mask = torch.zeros((B, N), dtype=torch.bool)  # No masking, all tokens are visible
model.eval()
with torch.no_grad():
  output = model.encoder(rnd_image)  # [B, N, C] N = 8 * 10 * 10

output = output.reshape(rnd_image.size(0),8,10,10,-1) 

In [3]:
model.encoder

PretrainVisionTransformerEncoder(
  (patch_embed): PatchEmbed(
    (proj): Conv3d(3, 512, kernel_size=(2, 16, 16), stride=(2, 16, 16))
  )
  (blocks): ModuleList(
    (0): LGBlock(
      (first_attn_norm0): LayerNorm((512,), eps=1e-06, elementwise_affine=True)
      (first_attn): GeneralAttention(
        (q): Linear(in_features=512, out_features=512, bias=False)
        (kv): Linear(in_features=512, out_features=1024, bias=False)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=512, out_features=512, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (second_attn_norm0): LayerNorm((512,), eps=1e-06, elementwise_affine=True)
      (second_attn): GeneralAttention(
        (q): Linear(in_features=512, out_features=512, bias=False)
        (kv): Linear(in_features=512, out_features=1024, bias=False)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=512, out_features=512, bias=True)
        (p